# 5-Minute Reliance LSTM Trading Pipeline (Triple-Barrier, R-Multiple)

Comprehensive, modular, GPU-ready (PyTorch) notebook for:
1. Data loading & preprocessing (2015–2021 train, 2022 test)
2. Feature engineering (RSI, MACD, Bollinger, ATR, EMAs, lags)
3. Triple-barrier labeling (target/stop/horizon)
4. LSTM model (sequence length = 60 candles)
5. Class-weighted training (CrossEntropy)
6. Evaluation (classification report, confusion matrix, prediction overlay)
7. Trade simulation (R-multiple risk, stop/target/momentum fade)
8. Performance stats (win rate, avg R, P&L, max drawdown, equity curve)
9. Live inference function (`predict_live`)

Classes: 1 = Long win (target first), -1 = Short win (stop first → profitable short), 0 = Neutral.

DISCLAIMER: Educational example only. Not investment advice.

In [ ]:
# ---- 0. Environment & Dependencies ----
# (Uncomment if running first time)
# %pip install pandas numpy scikit-learn matplotlib seaborn plotly ta torch torchvision torchaudio xgboost mplfinance --quiet

import warnings, os, math, json, time, gc, datetime as dt
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mplfinance as mpf
import plotly.graph_objects as go
from dataclasses import dataclass
from typing import List, Dict, Tuple

import ta
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")
if DEVICE=='cuda':
    print(torch.cuda.get_device_name(0))

In [ ]:
# The rest of the notebook cells will be appended programmatically below (do not edit this cell).

### 13. Next Steps / Enhancements
- Add walk-forward (purged) cross-validation
- Hyperparameter tuning (Optuna)
- Transaction costs & slippage modeling
- Meta-labeling & ensemble with tree models
- Regime detection (volatility clustering, HMM)
- Broker API integration (e.g., Zerodha Kite) for live execution

End of notebook.

In [ ]:
# ---- 12. Live Inference Function ----
@torch.no_grad()
def predict_live(latest_df: pd.DataFrame, model: nn.Module, scaler: StandardScaler, features: List[str], seq_len:int=60, risk: RiskConfig = RiskConfig()):
    assert len(latest_df) >= seq_len, 'Not enough bars for sequence.'
    engineered = add_features(latest_df)
    engineered = engineered.tail(seq_len)
    X = engineered[features].values.astype(np.float32)
    X_scaled = scaler.transform(X)
    tensor = torch.from_numpy(X_scaled).unsqueeze(0).to(DEVICE)
    model.eval(); logits = model(tensor)
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    p_short, p_neutral, p_long = probs[label_map[-1]], probs[label_map[0]], probs[label_map[1]]
    atr_last = engineered['atr_14'].iloc[-1]; close_last = engineered['close'].iloc[-1]
    if p_long>0.6 and atr_last>0:
        stop_dist = atr_last * risk.atr_mult
        stop = close_last - stop_dist; target = close_last + risk.r_multiple * stop_dist
        size = int(risk.capital*risk.risk_per_trade/stop_dist) if stop_dist>0 else 0
        return {'signal':'LONG','entry':float(close_last),'stop':float(stop),'target':float(target),'size':size,'probs':{'short':float(p_short),'neutral':float(p_neutral),'long':float(p_long)}}
    if p_short>0.6 and atr_last>0:
        stop_dist = atr_last * risk.atr_mult
        stop = close_last + stop_dist; target = close_last - risk.r_multiple * stop_dist
        size = int(risk.capital*risk.risk_per_trade/stop_dist) if stop_dist>0 else 0
        return {'signal':'SHORT','entry':float(close_last),'stop':float(stop),'target':float(target),'size':size,'probs':{'short':float(p_short),'neutral':float(p_neutral),'long':float(p_long)}}
    return {'signal':'HOLD','probs':{'short':float(p_short),'neutral':float(p_neutral),'long':float(p_long)}}

live_example = predict_live(test_df.tail(500), model, scaler, feature_cols, SEQ_LEN)
live_example

In [ ]:
# ---- 11. Visualization (Candles + Signals, Equity Curve, Prediction Overlay) ----
# Candlestick with entries/exits
plot_df = test_df.copy().iloc[:2000]  # subset if very large
plot_df_reset = plot_df.reset_index()
# Merge trade markers
entries = trades_df[['entry_time','entry_price','direction']].copy()
exits = trades_df[['exit_time','exit_price']].copy()

mpf_df = plot_df[['open','high','low','close','volume']].copy()
addplots=[]
# Markers
long_points = entries[entries.direction=='LONG']
short_points = entries[entries.direction=='SHORT']
if not long_points.empty:
    addplots.append(mpf.make_addplot(long_points.set_index('entry_time')['entry_price'], type='scatter', markersize=60, marker='^', color='green'))
if not short_points.empty:
    addplots.append(mpf.make_addplot(short_points.set_index('entry_time')['entry_price'], type='scatter', markersize=60, marker='v', color='red'))
if not exits.empty:
    addplots.append(mpf.make_addplot(exits.set_index('exit_time')['exit_price'], type='scatter', markersize=40, marker='x', color='orange'))

mpf.plot(mpf_df, type='candle', style='charles', addplot=addplots, volume=True, title='Candlestick with Trade Markers (subset)')

# Equity curve
if not trades_df.empty:
    trades_df['equity'] = trades_df['pnl'].cumsum()
    plt.figure(figsize=(6,3)); plt.plot(trades_df['exit_time'], trades_df['equity']); plt.title('Equity Curve'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

# Prediction vs Actual subset
subset = val_aligned.iloc[:1500]
plt.figure(figsize=(8,3)); plt.plot(subset.index, subset['label'], label='Actual', alpha=0.7)
plt.plot(subset.index, subset['pred_label'], label='Pred', alpha=0.7)
plt.legend(); plt.title('Predicted vs Actual Labels (subset index)'); plt.tight_layout(); plt.show()

In [ ]:
# ---- 10. Trade Performance Summary ----
def trade_stats(trades: pd.DataFrame) -> Dict[str,float]:
    if trades.empty:
        return {'num_trades':0,'win_rate':0,'avg_r':0,'total_profit':0,'max_drawdown':0}
    wins = trades['pnl']>0
    equity = trades['pnl'].cumsum()
    roll_max = equity.cummax(); drawdown = equity-roll_max; max_dd = drawdown.min()
    return {
        'num_trades': int(len(trades)),
        'win_rate': float(wins.mean()),
        'avg_r': float(trades['r_multiple'].mean()),
        'total_profit': float(trades['pnl'].sum()),
        'max_drawdown': float(max_dd)
    }

stats = trade_stats(trades_df)
trades_df.to_csv('simulated_trades.csv', index=False)
print(stats)
trades_df.head()

In [ ]:
# ---- 9. Trade Simulation (R-Multiple Risk Management) ----
def simulate_trades(test_full: pd.DataFrame, probs: np.ndarray, risk: RiskConfig = RiskConfig(), p_thresh=0.6):
    df_sim = test_full.iloc[SEQ_LEN:SEQ_LEN+len(probs)].copy()
    df_sim = df_sim.reset_index().rename(columns={'index':'date'})
    trades=[]; open_pos=False; direction=0; entry_i=None
    risk_amt = risk.capital * risk.risk_per_trade
    for i in range(len(df_sim)-risk.max_holding-1):
        row = df_sim.iloc[i]
        p_short = probs[i, label_map[-1]]; p_neutral = probs[i, label_map[0]]; p_long = probs[i, label_map[1]]
        if not open_pos:
            if p_long>p_thresh and row['atr_14']>0:
                entry=row['close']; sd=row['atr_14']*risk.atr_mult
                if sd<=0: continue
                size=int(risk_amt/sd);
                if size<=0: continue
                stop=entry-sd; target=entry+risk.r_multiple*sd
                trades.append({'entry_time':row['date'],'entry_price':entry,'direction':'LONG','stop':stop,'target':target,'size':size,'exit_time':None,'exit_price':None,'pnl':None,'r_multiple':None})
                open_pos=True; direction=1; entry_i=i
            elif p_short>p_thresh and row['atr_14']>0:
                entry=row['close']; sd=row['atr_14']*risk.atr_mult
                if sd<=0: continue
                size=int(risk_amt/sd);
                if size<=0: continue
                stop=entry+sd; target=entry-risk.r_multiple*sd
                trades.append({'entry_time':row['date'],'entry_price':entry,'direction':'SHORT','stop':stop,'target':target,'size':size,'exit_time':None,'exit_price':None,'pnl':None,'r_multiple':None})
                open_pos=True; direction=-1; entry_i=i
        else:
            t=trades[-1]; entry=t['entry_price']; stop=t['stop']; target=t['target']; size=t['size']; hold=i-entry_i
            hi=df_sim['high'].iloc[i]; lo=df_sim['low'].iloc[i]; rsi_now=df_sim['rsi_14'].iloc[i]
            exit_flag=False; exit_price=df_sim['close'].iloc[i]
            if direction==1:
                if hi>=target: exit_flag=True; exit_price=target
                elif lo<=stop: exit_flag=True; exit_price=stop
                elif rsi_now<50 or hold>=risk.max_holding: exit_flag=True
                if exit_flag:
                    pnl=(exit_price-entry)*size
                    r_mult=(exit_price-entry)/(entry-stop) if (entry-stop)!=0 else 0
            else:
                if lo<=target: exit_flag=True; exit_price=target
                elif hi>=stop: exit_flag=True; exit_price=stop
                elif rsi_now>50 or hold>=risk.max_holding: exit_flag=True
                if exit_flag:
                    pnl=(entry-exit_price)*size
                    r_mult=(entry-exit_price)/(stop-entry) if (stop-entry)!=0 else 0
            if exit_flag:
                t['exit_time']=df_sim['date'].iloc[i]; t['exit_price']=exit_price; t['pnl']=pnl; t['r_multiple']=r_mult
                open_pos=False; direction=0; entry_i=None
    trades_df=pd.DataFrame(trades)
    trades_df=trades_df.dropna(subset=['exit_price']).reset_index(drop=True)
    return trades_df

trades_df = simulate_trades(test_df, all_probs, RiskConfig(), p_thresh=0.6)
print(trades_df.head())
print('Trades:', len(trades_df))

In [ ]:
# ---- 8. Evaluation (Classification Report & Confusion Matrix) ----
chkpt = torch.load('lstm_trading_model.pth', map_location=DEVICE)
model.load_state_dict(chkpt['model_state'])
model.eval()
all_preds=[]; all_true=[]; all_probs=[]
with torch.no_grad():
    for X,y in val_loader:
        X=X.to(DEVICE); y=y.to(DEVICE)
        logits = model(X)
        probs = torch.softmax(logits, dim=1)
        all_preds.append(probs.argmax(1).cpu().numpy())
        all_true.append(y.cpu().numpy())
        all_probs.append(probs.cpu().numpy())
all_preds = np.concatenate(all_preds); all_true = np.concatenate(all_true); all_probs=np.concatenate(all_probs)
true_dec = np.vectorize(inv_label_map.get)(all_true)
pred_dec = np.vectorize(inv_label_map.get)(all_preds)
print(classification_report(true_dec, pred_dec, digits=4))
cm = confusion_matrix(true_dec, pred_dec, labels=[-1,0,1])
plt.figure(figsize=(4,3)); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['-1','0','1'], yticklabels=['-1','0','1']); plt.title('Confusion Matrix'); plt.ylabel('True'); plt.xlabel('Pred'); plt.tight_layout(); plt.show()
# Align with test_df rows
val_indices = np.arange(SEQ_LEN, SEQ_LEN + len(all_preds))
val_aligned = test_df.iloc[val_indices].copy()
val_aligned['pred_label'] = pred_dec
val_aligned.head()

In [ ]:
# ---- 7. Training Loop ----
EPOCHS = 25
lr = 1e-3
# Class weights
train_label_counts = pd.Series(train_df['label'].map(label_map)).value_counts().sort_index()
weights = train_label_counts.sum() / (train_label_counts * len(train_label_counts))
class_weights = torch.tensor(weights.values, dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

history={'train_loss':[],'val_loss':[]}

@torch.no_grad()
def evaluate(loader):
    model.eval(); total=0; loss_sum=0; correct=0
    for X,y in loader:
        X=X.to(DEVICE); y=y.to(DEVICE)
        out = model(X)
        loss = criterion(out,y)
        loss_sum += loss.item()*y.size(0)
        preds = out.argmax(1)
        correct += (preds==y).sum().item(); total += y.size(0)
    return loss_sum/total, correct/total

best_val=float('inf')
for epoch in range(1,EPOCHS+1):
    model.train(); run_loss=0; total=0
    for X,y in train_loader:
        X=X.to(DEVICE); y=y.to(DEVICE)
        optimizer.zero_grad(); out=model(X); loss=criterion(out,y); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step(); run_loss+=loss.item()*y.size(0); total+=y.size(0)
    tr_loss=run_loss/total; val_loss,val_acc=evaluate(val_loader)
    history['train_loss'].append(tr_loss); history['val_loss'].append(val_loss)
    if val_loss<best_val:
        best_val=val_loss
        torch.save({'model_state':model.state_dict(),'scaler':scaler,'feature_cols':feature_cols}, 'lstm_trading_model.pth')
    if epoch%5==0 or epoch==1:
        print(f'Epoch {epoch}/{EPOCHS} train_loss {tr_loss:.4f} val_loss {val_loss:.4f} val_acc {val_acc:.3f}')

plt.figure(figsize=(6,3)); plt.plot(history['train_loss'],label='train'); plt.plot(history['val_loss'],label='val'); plt.legend(); plt.title('Loss'); plt.show()

In [ ]:
# ---- 6. PyTorch LSTM Model ----
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden=64, layers=2, dropout=0.3, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, num_layers=layers, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, num_classes)
    def forward(self, x):
        out,_ = self.lstm(x)
        last = out[:,-1,:]
        return self.fc(self.dropout(last))

model = LSTMClassifier(len(feature_cols)).to(DEVICE)
model

In [ ]:
# ---- 5. Sequence Dataset (Past 60 Candles) ----
SEQ_LEN = 60
label_map = {-1:0, 0:1, 1:2}
inv_label_map = {v:k for k,v in label_map.items()}

class SeqDataset(Dataset):
    def __init__(self, df: pd.DataFrame, features: List[str], seq_len: int=60):
        self.df = df.copy()
        self.X = self.df[features].values.astype(np.float32)
        self.y = self.df['label'].map(label_map).values.astype(int)
        self.seq_len=seq_len
    def __len__(self):
        return len(self.df)-self.seq_len
    def __getitem__(self, idx):
        Xseq = self.X[idx:idx+self.seq_len]
        y_t = self.y[idx+self.seq_len]
        return torch.from_numpy(Xseq), torch.tensor(y_t, dtype=torch.long)

train_ds = SeqDataset(train_df, feature_cols, SEQ_LEN)
test_ds = SeqDataset(test_df, feature_cols, SEQ_LEN)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, drop_last=True)
val_loader = DataLoader(test_ds, batch_size=128, shuffle=False)
len(train_ds), len(test_ds)

In [ ]:
# ---- 4. Train/Test Split & Scaling (2015-2021 train, 2022 test) ----
train_start = pd.Timestamp('2015-01-01')
train_end = pd.Timestamp('2021-12-31 23:59:59')
TEST_START = pd.Timestamp('2022-01-01')
train_df = labeled[(labeled.index>=train_start) & (labeled.index<=train_end)].copy()
test_df = labeled[(labeled.index>=TEST_START)].copy()
feature_cols = [c for c in labeled.columns if c not in ['label']]
scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols] = scaler.transform(test_df[feature_cols])
len(train_df), len(test_df), feature_cols[:8]

In [ ]:
# ---- 3. Triple-Barrier Labeling ----
@dataclass
class RiskConfig:
    capital: float = 100_000.0
    risk_per_trade: float = 0.01
    atr_mult: float = 1.5
    r_multiple: float = 2.0
    max_holding: int = 50


def triple_barrier(df: pd.DataFrame, risk: RiskConfig = RiskConfig()) -> pd.DataFrame:
    d = df.copy()
    n = len(d)
    labels = np.zeros(n, dtype=int)
    closes = d['close'].values; highs=d['high'].values; lows=d['low'].values
    atr = d['atr_14'].values; rsi = d['rsi_14'].values
    H = risk.max_holding
    for i in range(n - H - 1):
        stop_dist = atr[i] * risk.atr_mult
        if not np.isfinite(stop_dist) or stop_dist<=0: continue
        entry = closes[i]
        upper = entry + risk.r_multiple * stop_dist
        lower = entry - stop_dist
        label = 0
        for fwd in range(1, H+1):
            hi = highs[i+fwd]; lo = lows[i+fwd]
            if hi >= upper: label = 1; break
            if lo <= lower: label = -1; break
            if (rsi[i] > 50 and rsi[i+fwd] < 50) or (rsi[i] < 50 and rsi[i+fwd] > 50):
                label = 0; break
        labels[i] = label
    d['label'] = labels
    return d

labeled = triple_barrier(feat)
print(labeled['label'].value_counts())
labeled.head()

In [ ]:
# ---- 2. Feature Engineering (Indicators & Lags) ----

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d['return'] = d['close'].pct_change()
    d['log_return'] = np.log(d['close']).diff()
    # EMAs
    d['ema_20'] = ta.trend.EMAIndicator(d['close'], window=20).ema_indicator()
    d['ema_50'] = ta.trend.EMAIndicator(d['close'], window=50).ema_indicator()
    d['ema_200'] = ta.trend.EMAIndicator(d['close'], window=200).ema_indicator()
    # RSI
    d['rsi_14'] = ta.momentum.RSIIndicator(d['close'], window=14).rsi()
    # MACD
    macd = ta.trend.MACD(d['close'], window_slow=26, window_fast=12, window_sign=9)
    d['macd'] = macd.macd(); d['macd_signal'] = macd.macd_signal(); d['macd_hist'] = macd.macd_diff()
    # ATR
    atr = ta.volatility.AverageTrueRange(d['high'], d['low'], d['close'], window=14)
    d['atr_14'] = atr.average_true_range()
    # Bollinger
    bb = ta.volatility.BollingerBands(d['close'], window=20, window_dev=2)
    d['bb_high'] = bb.bollinger_hband(); d['bb_low'] = bb.bollinger_lband(); d['bb_mid'] = bb.bollinger_mavg()
    d['bb_width'] = (d['bb_high'] - d['bb_low']) / d['bb_mid']
    # Lags
    for lag in [3,5,10]:
        d[f'return_lag_{lag}'] = d['return'].shift(lag)
    # Relative levels
    d['close_over_ema20'] = d['close']/d['ema_20'] - 1
    d['close_over_ema50'] = d['close']/d['ema_50'] - 1
    d['close_over_ema200'] = d['close']/d['ema_200'] - 1
    d.dropna(inplace=True)
    return d

feat = add_features(raw)
print('Features shape:', feat.shape)
feat.head()

In [ ]:
# ---- 1. Data Loading & Preprocessing ----
CSV_PATH = 'RELIANCE_5minute.csv'
raw = pd.read_csv(CSV_PATH)
assert set(['date','open','high','low','close','volume']).issubset(raw.columns)
raw['date'] = pd.to_datetime(raw['date'])
raw = raw.sort_values('date').reset_index(drop=True)
raw.set_index('date', inplace=True)
print(raw.head())
print('Data span:', raw.index.min(), '->', raw.index.max(), 'rows:', len(raw))